# Lecția 13 - Memoria agentului


## Configurare

Acest caiet demonstrează cum să construiești un agent de rezervări de călătorie cu **memorie persistentă** folosind **Microsoft Agent Framework** (MAF).

Vei învăța cum diferitele tipuri de memorie ale agentului — de lucru, pe termen scurt și pe termen lung — afectează modul în care un agent reține și folosește informațiile pe parcursul conversațiilor.

**Prerechizite:**
- Un proiect Microsoft Foundry cu un model de chat implementat (de ex. `gpt-5-mini`).
- Să fii autentificat cu Azure CLI — rulează `az login` în terminalul tău.
- `AZURE_AI_PROJECT_ENDPOINT` — endpoint-ul proiectului tău Microsoft Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — numele modelului tău implementat.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Tipuri de Memorie ale Agentului

Agenții AI pot folosi diferite tipuri de memorie, fiecare servind unui scop distinct:

### Memorie de Lucru
Firul conversației în sine — mesajele schimbate într-o singură sesiune. Agentul se poate referi la mesajele anterioare din același fir pentru a menține coerența. În MAF aceasta este creată prin **`agent.create_session()`**, care returnează un `AgentSession`.

### Memorie pe Termen Scurt
Informații care persistă pe durata unei sarcini sau sesiuni, dar nu sunt stocate permanent. De exemplu, agentul poate acumula fapte în timpul unei conversații de planificare pe mai multe runde și le poate folosi pentru a produce un itinerariu final.

### Memorie pe Termen Lung
Preferințe și fapte care persistă **peste sesiuni**. Un utilizator care revine nu ar trebui să fie nevoit să repete restricțiile alimentare sau stilul de călătorie. Memoria pe termen lung este de obicei susținută de un magazin extern — o bază de date, fișier sau index vectorial — și este pusă la dispoziția agentului prin instrumente.


## Memoria de lucru cu sesiuni

Cea mai simplă formă de memorie este sesiunea de conversație. Când transmiți aceeași sesiune (creată prin `agent.create_session()`) la apeluri succesive `agent.run()`, agentul vede istoricul complet al acelei conversații și poate reaminti detalii anterioare.

Să creăm un agent de călătorii și să demonstrăm memoria de lucru.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agentul și-a amintit corect bugetul deoarece ambele mesaje împart aceeași sesiune. Aceasta este **memorie de lucru** — există doar pe durata sesiunii.

### Ce se întâmplă cu un fir nou?

Dacă creăm o sesiune **nouă**, agentul nu are memorie despre conversația anterioară:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Tipar pentru Memorie pe Termen Lung

Pentru a reține preferințele utilizatorului **pe parcursul mai multor sesiuni**, avem nevoie de un depozit persistent care să existe în afara firului conversației. Agentul accesează acest depozit prin **unelte** — funcții pe care le poate apela pentru a salva și recupera informații.

Mai jos implementăm un depozit simplu de preferințe în memorie (în producție ai folosi o bază de date sau un index vectorial) și îl expunem ca unelte pe care agentul să le poată folosi.

### Arhitectură
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenariul 1 — Utilizator pentru prima dată rezervă o excursie aniversară

Sarah vizitează pentru prima dată. Agentul ar trebui să-i stocheze preferințele prin intermediul uneltelor și să le folosească pentru a recomanda hoteluri.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenariul 2 — Sarah revine după săptămâni

Sarah începe un **fir complet nou** (simulând o sesiune nouă). Memoria de lucru este goală, dar depozitul de preferințe pe termen lung conține încă informațiile ei. Agentul ar trebui să le recupereze și să le folosească pentru a personaliza recomandările.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Rezumat

În această lecție ai învățat trei tipuri de memorie pentru agenți și cum să le implementezi cu Microsoft Agent Framework:

| Tip de Memorie | Mecanism MAF | Durată |
|---|---|---|
| **De lucru** | `agent.create_session()` | O singură conversație |
| **Pe termen scurt** | Context acumulat într-un fir | O singură sarcină / sesiune |
| **Pe termen lung** | Magazin extern accesat prin funcții `@tool` | Între sesiuni |

### Puncte cheie
1. **`agent.create_session()`** oferă memorie de lucru — agentul vede istoricul complet al conversației într-o sesiune.
2. **Sesiunile noi pierd contextul** — fără memorie pe termen lung, agentul nu-și poate aminti conversațiile trecute.
3. **Funcțiile `@tool`** fac legătura — acestea permit agentului să salveze și să recupereze informații dintr-un depozit persistent.
4. **Personalizarea se îmbunătățește în timp** — cu cât se stochează mai multe preferințe, cu atât recomandările agentului sunt mai bune.

### Aplicații în lumea reală
- **Serviciul Clienți**: Amintirea istoricului și preferințelor clientului
- **Asistenți personali**: Menținerea contextului pe durata zilelor sau săptămânilor
- **Îngrijire medicală**: Urmărirea informațiilor și preferințelor pacienților
- **Comerț electronic**: Cumpărături personalizate bazate pe istoric

### Pași următori
- Înlocuiește dicționarul în memorie cu o bază de date sau magazin vectorial (ex. Azure AI Search)
- Adaugă expirarea memoriei pentru informații sensibile în timp
- Construiește sisteme multi-agent cu memorie partajată
- Explorează notebook-ul Cognee pentru memorie susținută de grafuri de cunoștințe


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
